# Prepare the ERA5 file for the downscaling notebook

Cuts hourly ERA5 convective and large-scale rain for Typhoon Nando (20 to 24 Sep 2025) over northern Luzon
from Google's public ARCO-ERA5 copy (no account needed) and saves it as `nando_era5.nc` (a few MB).
Run once, then commit the file as `data/nando_era5.nc`. The participant notebook downloads it from there.

In [ ]:
%pip install -q gcsfs zarr
import xarray as xr

ds = xr.open_zarr("gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3",
                  chunks=None, storage_options=dict(token="anon"))
print("ERA5 available until:", ds.attrs.get("valid_time_stop"))

In [ ]:
sub = (ds[["convective_precipitation", "large_scale_precipitation"]]
       .sel(time=slice("2025-09-20T00", "2025-09-24T00"), latitude=slice(26, 8), longitude=slice(112, 130))
       .rename(convective_precipitation="cp", large_scale_precipitation="lsp", latitude="lat", longitude="lon")
       .load())
assert sub.sizes["time"] == 97 and int(sub.isnull().sum().to_array().sum()) == 0
enc = {v: {"zlib": True, "complevel": 4, "dtype": "float32"} for v in sub.data_vars}
sub.to_netcdf("nando_era5.nc", encoding=enc)
print(sub)
print("Max hourly rain (mm):", float((sub.cp + sub.lsp).max()) * 1000)

In [ ]:
from google.colab import files
files.download("nando_era5.nc")